In [1]:
from analysis import *

In [2]:
cities = (
    "Atlanta,Salt Lake City,San Francisco,New York City".lower()
    .replace(" ", "_")
    .split(",")
)

# cities = "Salt Lake City".lower().replace(" ", "_").split(",")

for city in cities:
    print(f"starting for {city}")
    gtfs = GTFS(city)
    x = gtfs.run_delay_analyzer()
    break

starting for atlanta
Deleted previous records:  0


In [24]:
def actual_arrivals_for_trip(trip_updates):
    x = trip_updates

    if "trip__schedule_relationship" in x.columns:
        mask = x.trip__schedule_relationship == "CANCELED"
        x = x.loc[~mask]
        if x.shape[0] == 0:
            return None

    dates = x.date.unique()
    print(dates)
    stops = np.unique(
        x.loc[:, x.columns.str.contains("__stop_sequence")].values.flatten()
    )
    stops = stops[~np.isnan(stops)]
    x = x.loc[
        :, ~x.map(lambda x: np.nan if x == "NONE" else x).isnull().all(axis=0)
    ]

    aatp = {}
    for date in dates:
        x_day = x.loc[x.date == date]
        stop_seqs = x_day.loc[:, x_day.columns.str.contains("__stop_sequence")]

        for i in range(stop_seqs.shape[1]):
            s = stop_seqs[f"stop_time_update_{i}__stop_sequence"]
            pair = s.iloc[::-1].drop_duplicates(keep="first").to_dict()
            pair = {v: k for k, v in pair.items()}
            for k in pair.keys():
                if (
                    f"stop_time_update_{i}__schedule_relationship" in x_day.columns
                    and x_day.loc[
                        pair[k], f"stop_time_update_{i}__schedule_relationship"
                    ]
                    == "SKIPPED"
                ):
                    continue
                if (k, date) not in aatp.keys() and not np.isnan(k):
                    aatp[(k, date)] = (pair[k], i, x_day.loc[pair[k], "time"].item())

    aatp = pd.DataFrame(aatp).T.reset_index()
    aatp = aatp.rename(
        columns={
            k: v
            for k, v in zip(
                aatp.columns, ["stop_sequence", "date", "ind", "col", "time"]
            )
        }
    )

    aat = []
    for row in aatp.itertuples():
        try:
            aat.append(
                {
                    "stop_sequence": row.stop_sequence,
                    "time": row.time,
                    "date": row.date,
                    "actual_arrival_time": x.loc[
                        row.ind, f"stop_time_update_{row.col}__arrival__time"
                    ].item(),
                }
            )
        except:
            aat.append(
                {
                    "stop_sequence": row.stop_sequence,
                    "time": row.time,
                    "date": row.date,
                    "actual_arrival_time": x.loc[
                        row.ind, f"stop_time_update_{row.col}__departure__time"
                    ].item(),
                }
            )

    aat = pd.DataFrame(aat)

    if aat.shape[0] == 0:
        return None

    aat["actual_arrival_time_unix"] = aat["actual_arrival_time"]

    aat = (
        aat.sort_values("stop_sequence", ignore_index=True)
        .astype({"actual_arrival_time": "datetime64[s]"})
        .assign(
            actual_arrival_time=lambda x: x.actual_arrival_time.dt.tz_localize(
                "UTC"
            )
        )
    )

    return aat

In [25]:
actual_arrivals_for_trip(x)

['2025-11-08' '2025-11-15']


,stop_sequence,time,date,actual_arrival_time,actual_arrival_time_unix
0,1.0,1762598111,2025-11-08,2025-11-08 10:34:40+00:00,1.762598e+09
1,2.0,1762598831,2025-11-08,2025-11-08 10:35:25+00:00,1.762598e+09
2,2.0,1763203590,2025-11-15,2025-11-15 10:34:11+00:00,1.763203e+09
3,3.0,1762598831,2025-11-08,2025-11-08 10:35:50+00:00,1.762598e+09
4,3.0,1763203590,2025-11-15,2025-11-15 10:34:36+00:00,1.763203e+09
...,...,...,...,...,...
70,36.0,1763205690,2025-11-15,2025-11-15 11:16:33+00:00,1.763205e+09
71,37.0,1762599731,2025-11-08,2025-11-08 11:02:31+00:00,1.762600e+09
72,37.0,1763205690,2025-11-15,2025-11-15 11:17:09+00:00,1.763205e+09
73,38.0,1762599731,2025-11-08,2025-11-08 11:04:17+00:00,1.762600e+09
